# RAG с использованием библиотеки LangChain

В этом тюториале мы создадим полноценную RAG-систему с помощью библиотеки _langchain_.

Код и документация библиотек семейства _LangChain_: https://github.com/langchain-ai

Эти библиотеки можно поставить в виртуальное окружение командой `pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-openai`

Импортируем модули которые нам понадобятся впоследствии:

In [1]:
import logging
from timeit import default_timer as timer

import datasets
import sentence_transformers as sbert

from langchain_classic.chains import RetrievalQA
from langchain_classic.chains import LLMChain
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

Подготовим логгер:

In [2]:
logging.basicConfig(level=logging.INFO)

## Датасет

Будем работать с простым датасетом RAG Dataset 1200.

Этот датасет удобен тем, что:
- он небольшой, всего 1200 объектов
- содержит развернутые ответы (а не короткие как многие стандартные датасеты для RAG)
- в нем есть эталонные ответы, которые можно использовать для оценки качества

Ссылка: https://huggingface.co/datasets/neural-bridge/rag-dataset-1200

Загрузим датасет:

In [3]:
start = timer()
data = datasets.load_dataset("neural-bridge/rag-dataset-1200")
logging.info(f"loaded dataset: elapsed = {timer() - start:.3f}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/neural-bridge/rag-dataset-1200/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/neural-bridge/rag-dataset-1200/abef6f6c7bf300e0ee16b6ff1430393afdc751a2/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/neural-bridge/rag-dataset-1200/resolve/abef6f6c7bf300e0ee16b6ff1430393afdc751a2/rag-dataset-1200.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/neural-bridge/rag-dataset-1200/neural-bridge/rag-dataset-1200.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/neural-bridge/rag-dataset-1200/revision/abef6f6c7bf300e0ee16b6ff1430393afdc751a2 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/neural-bridge/rag-dataset-1200/resolve/abef6f6c7bf300e0ee16b6ff1430393

Проведем EDA:

In [4]:
print(data)

DatasetDict({
    train: Dataset({
        features: ['context', 'question', 'answer'],
        num_rows: 960
    })
    test: Dataset({
        features: ['context', 'question', 'answer'],
        num_rows: 240
    })
})


Посмотрим на один из примеров:

In [5]:
example_idx = 450

data_train = data['train']
example = data_train[450]
print(example.keys())

dict_keys(['context', 'question', 'answer'])


Посмотрим на контекст:

In [6]:
print(example['context'])

Things to do around Barcelona
Barcelona, a part of the charm and beauty of this city is that it is surrounded by many wonderful places that you can visit, especially if you’ve already been to the city a few times before and are looking for something new. If you’ve never been to the Catalan capital before then it may be best that you check out the Barcelona accommodation on offer in the city center and plan a trip within the city walls. Otherwise, to get the most out of your trip, you can rent an apartment that will enable you to see the best of the inner city and the surrounding area by renting a bit further from the center. You’ll save money too as quality apartments away from the center are cheaper to stay in.
Girona
The city of Girona is 100km from Barcelona and was founded as in 77BC, as such, this beautiful place has a rich history. You can stroll through the picturesque streets around the center and see the cathedral with the largest gothic nave in the world. You can also take ph

Посмотрим на запрос и эталонный ответ:

In [7]:
print(f"Question: {example['question']}")
print(f"Answer: {example['answer']}")

Question: What are some of the places to visit around Barcelona?
Answer: Some of the places to visit around Barcelona include Girona, Costa Brava, Figueres, Sitges, Montaña de Montserrat, and La Montaña de la sal – Cardona.


## Индексация

Распакуем датасет, и сохраним все документы, запросы и эталонные ответы:

In [8]:
# Unpack dataset and collect docs/queries
docs = []
queries = []
answers = []
for i, example in enumerate(data_train):
    context = example["context"]
    query = example["question"]
    answer = example["answer"]
    docs.append(Document(page_content=context))
    queries.append(query)
    answers.append(answer)
logging.info(f"collected: num_docs = {len(docs)}")

INFO:root:collected: num_docs = 960


Разобъем текст на чанки с использованием стандартного _RecursiveCharacterTextSplitter_.

In [9]:
# Make chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)
logging.info(f"splitted into chunks: num_chunks = {len(chunks)}")

INFO:root:splitted into chunks: num_chunks = 11881


Подготовим эмбеддер, который будет использовать GPU, и заэмбеддим наши документы.

Для этого у нас должен быть установлен пакет _sentence_transformers_: ```pip install sentence-transformers```

Скачаем и загрузим модель _MiniLM_:

In [10]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

# Load embedder
embeddings = HuggingFaceEmbeddings(model_name=model_name, model_kwargs = {'device': 'cuda'})
logging.info(f"loaded model: model_name = {model_name}")

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Tempor

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Re

Заэмбеддим все чанки, и сохраним эмбеддинги в _faiss_.

Для этого у нас должен быть установлен пакет _faiss_: ```pip install faiss-cpu```

In [11]:
# Embed all the chunks
start = timer()
vectorstore = FAISS.from_documents(chunks, embeddings)
logging.info(f"embedded chunks: elapsed = {timer() - start:.3f}")

INFO:faiss.loader:Loading faiss with AVX2 support.
INFO:faiss.loader:Successfully loaded faiss with AVX2 support.
INFO:root:embedded chunks: elapsed = 4.577


При желании, этот индекс теперь можно было бы сохранить на диск.

## Поиск и ранжирование

Теперь можно попробовать найти в нашем индексе релевантные запросу документы.

Сначала нам придется создать объект-ретривер.

Существуют разные ретриверы:
- dense-ретриверы для поиска по эмбеддингам
- лексические для поиска по ключевым в классическом обратном индексе
- гибридные, которые сочетают достоинства обоих подходов

Мы, для простоты, будем использовать обычный dense.

Найдем в нашем faiss k=10 наиболее близких по косинусному расстоянию документов:

In [12]:
# Initialize dense retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
logging.info(f"created retriever")

INFO:root:created retriever


Теперь поднимем с его помощью документы по запросу из нашего примера:

In [13]:
query = queries[example_idx]
print(f"QUERY: {query}")

# Retrieve documents
retrieved_docs = retriever.invoke(query)
for i, doc in enumerate(retrieved_docs):
    print(f"RETRIEVED: i = {i}")
    print(doc.page_content)

QUERY: What are some of the places to visit around Barcelona?
RETRIEVED: i = 0
Things to do around Barcelona
RETRIEVED: i = 1
Costa Brava
One of the best places to visit across Barcelona, full of an array of destinations. This region consists of 200 miles of coast north of Barcelona, from Blanes to Portbou. There’s a varied landscape too, from small fishing villages to medieval towns and mid-sized cities. If you had to choose only 2 places to visit though, then it’s recommended you check out Tossa de Mar and Roses.
RETRIEVED: i = 2
Barcelona, a part of the charm and beauty of this city is that it is surrounded by many wonderful places that you can visit, especially if you’ve already been to the city a few times before and are looking for something new. If you’ve never been to the Catalan capital before then it may be best that you check out the Barcelona accommodation on offer in the city center and plan a trip within the
RETRIEVED: i = 3
Montaña de Montserrat
The mountain of Montserra

Будем переранжировать поднятые документы с помощью модели кросс-энкодера.

Загрузим стандартный кросс-энкодер на основе _MiniLM_, дообученный на датасете _MS MARCO_:

In [14]:
ce_model_name = "cross-encoder/ms-marco-MiniLM-L6-v2"

# Load reranker
reranker = sbert.CrossEncoder(ce_model_name)
logging.info(f"loaded reranker model: model_name = {ce_model_name}")

INFO:sentence_transformers.base.model:No device provided, using cuda:0
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
INFO:sentence_transformers.base.model:No modules.json found for cross-encoder/ms-marco-MiniLM-L6-v2, initializing a new CrossEncoder model.
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https:

Напишем функцию для переранжирования _top_k_ документов с помощью нашего реранкера:

In [15]:
def rerank(query, docs, top_k=3):
    pairs = [(query, doc.page_content) for doc in docs]
    scores = reranker.predict(pairs)

    ranked_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked_docs[:top_k]]

Получим топы документов двумя способами, с переранжированием и без него:

In [16]:
top_k = 3

# Without reranking
top_docs = retrieved_docs[:top_k]
top_docs

[Document(id='b7deb9d3-02a9-46e7-8792-f87dd14a5b38', metadata={}, page_content='Things to do around Barcelona'),
 Document(id='1e14028e-3f77-4b5e-8449-3cb2e7886d41', metadata={}, page_content='Costa Brava\nOne of the best places to visit across Barcelona, full of an array of destinations. This region consists of 200 miles of coast north of Barcelona, from Blanes to Portbou. There’s a varied landscape too, from small fishing villages to medieval towns and mid-sized cities. If you had to choose only 2 places to visit though, then it’s recommended you check out Tossa de Mar and Roses.'),
 Document(id='6240d640-3461-4012-bfe6-c174b8b01ab1', metadata={}, page_content='Barcelona, a part of the charm and beauty of this city is that it is surrounded by many wonderful places that you can visit, especially if you’ve already been to the city a few times before and are looking for something new. If you’ve never been to the Catalan capital before then it may be best that you check out the Barcelona

In [17]:
# With reranking
reranked_top_docs = rerank(query, retrieved_docs, top_k=top_k)
reranked_top_docs

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Document(id='834045ae-b402-4d62-82bc-48f6a580eaa9', metadata={}, page_content='Montaña de Montserrat\nThe mountain of Montserrat is one of the most interesting places to visit near Barcelona. It’s practically an icon of the city, and of Catalonia too. Here you will find the famous monastery which houses a statue of the Virgin Mary, black, the patron saint of Catalonia.\nLa Montaña de la sal – CardonaBarcelona surrounding areas'),
 Document(id='28ac4c90-6245-4018-8e16-7c086e4a0297', metadata={}, page_content='Sitges\nThe town of Sitges is another place to visit near Barcelona. There are many medieval elements of the town that are still preserved and it has a rich cultural heritage because it was a landmark place for modernist intellectuals.\nMontaña de Montserrat'),
 Document(id='1e14028e-3f77-4b5e-8449-3cb2e7886d41', metadata={}, page_content='Costa Brava\nOne of the best places to visit across Barcelona, full of an array of destinations. This region consists of 200 miles of coast nor

## Генерация ответа

Теперь перейдем к самому интересному: используем найденные документы, чтобы сгенерить ответ.

Сначала нам придется подключиться к LLM.

Считаем, что LLM уже поднята с помощью скрипта `llm.sh`.

Давайте в этом убедимся:

In [18]:
!nvidia-smi

Thu Apr 30 20:20:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.58.03              Driver Version: 595.58.03      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 Ti     On  |   00000000:01:00.0 Off |                  N/A |
|  0%   40C    P2             21W /  165W |   10139MiB /  16380MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Для взаимодействия с LLM будем использовать стандартный OpenAI API.

Создадим клиента:

In [19]:
llm_model_name = "Qwen/Qwen3-4B-FP8"

# Create client
llm = ChatOpenAI(openai_api_base=f"http://localhost:8000/v1",
                 openai_api_key="EMPTY",
                 model_name=llm_model_name)
logging.info(f"created LLM client: model_name = {llm_model_name}")

INFO:root:created LLM client: model_name = Qwen/Qwen3-4B-FP8


Создадим шаблон для промпта, в который можно будет поставлять наш запрос и контекст (найденные документы):

In [20]:
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="Use the context below to answer the question.\nContext: {context}\nQuestion: {question}\nAnswer:")

Создадим объект-чейн, с помощью которого будем делать запросы к LLM:

In [21]:
chain = LLMChain(llm=llm, prompt=prompt)
logging.info(f"created LLM chain")

/tmp/ipykernel_51428/1126029735.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)
INFO:root:created LLM chain


Поместим поднятые документы в контекст:

In [22]:
# From retrieved docs
context = "\n\n".join([doc.page_content for doc in top_docs])

# From retrieved & reranked docs
reranked_context = "\n\n".join([doc.page_content for doc in reranked_top_docs])

Напишем вспомогательную функцию, которая распечатывает ответ LLM:

In [23]:
def print_rag_answer(query, context, ref_answer, rag_answer):
    # Golden reference answer
    print(f"\n*** Query:\n{query}")
    print(f"\n*** Context:\n{context}")
    print(f"\n*** RAG answer:\n{rag_answer}")
    print(f"\n*** Reference answer:\n{ref_answer}")

Cначала попробуем подать запрос вообще без контекста, и посмотрим что ответит модель.

Тут придется немного подождать пока LLM думает. При желании, можно перейти в консоль и понаблюдать за логами _vllm_.

In [24]:
query = queries[example_idx]
ref_answer = answers[example_idx]

# Make an LLM call
result = chain.invoke({"context": "", "question": query})
rag_answer = result["text"]
print_rag_answer(query, "", ref_answer, rag_answer)

INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"



*** Query:
What are some of the places to visit around Barcelona?

*** Context:


*** RAG answer:


Around Barcelona, some of the top places to visit include:  
- **Sagrada Família**: A iconic Gothic cathedral designed by Antoni Gaudí.  
- **Park Güell**: A vibrant public park with colorful mosaics and terraces.  
- **Gothic Quarter (Barri Gòtic)**: A historic district with medieval architecture and narrow streets.  
- **Barceloneta Beach**: A popular coastal area for swimming, sunbathing, and nightlife.  
- **Montjuïc**: Home to the Camp Nou stadium, the Museum of the Sea, and the Montjuïc Castle.  
- **Picasso Museum**: Features works by the famous artist, including his famous *Guernica*.  
- **Montseny Mountains**: A nature reserve offering hiking trails and scenic views.  
- **Parc de la Ciutadella**: A large park with a museum, fountains, and green spaces.  
- **Tibidabo Hill**: A hill with a cable car ride and panoramic city views.  
- **El Raval District**: Known for its vibran

Видно, что модель отвечает из общих знаний, и, в принципе, отвечает неплохо.

Что будет если мы теперь передадим ей полноценный контекст?
Начнем с варианта без переранжирования:

In [25]:
# Make an LLM call with retrieved context
result = chain.invoke({"context": context, "question": query})
rag_answer = result["text"]
print_rag_answer(query, context, ref_answer, rag_answer)

INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"



*** Query:
What are some of the places to visit around Barcelona?

*** Context:
Things to do around Barcelona

Costa Brava
One of the best places to visit across Barcelona, full of an array of destinations. This region consists of 200 miles of coast north of Barcelona, from Blanes to Portbou. There’s a varied landscape too, from small fishing villages to medieval towns and mid-sized cities. If you had to choose only 2 places to visit though, then it’s recommended you check out Tossa de Mar and Roses.

Barcelona, a part of the charm and beauty of this city is that it is surrounded by many wonderful places that you can visit, especially if you’ve already been to the city a few times before and are looking for something new. If you’ve never been to the Catalan capital before then it may be best that you check out the Barcelona accommodation on offer in the city center and plan a trip within the

*** RAG answer:


Some of the places to visit around Barcelona include the **Costa Brava** re

Видим, что модель теперь действительно опирается в своем ответе не только на общие знания, но и на контекст!

Аналогично, посмотрим на ответ с контекстом, полученным из переранжированных документов:

In [26]:
# Make an LLM call with retrieved context
result = chain.invoke({"context": reranked_context, "question": query})
rag_answer = result["text"]
print_rag_answer(query, reranked_context, ref_answer, rag_answer)

INFO:httpx:HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 200 OK"



*** Query:
What are some of the places to visit around Barcelona?

*** Context:
Montaña de Montserrat
The mountain of Montserrat is one of the most interesting places to visit near Barcelona. It’s practically an icon of the city, and of Catalonia too. Here you will find the famous monastery which houses a statue of the Virgin Mary, black, the patron saint of Catalonia.
La Montaña de la sal – CardonaBarcelona surrounding areas

Sitges
The town of Sitges is another place to visit near Barcelona. There are many medieval elements of the town that are still preserved and it has a rich cultural heritage because it was a landmark place for modernist intellectuals.
Montaña de Montserrat

Costa Brava
One of the best places to visit across Barcelona, full of an array of destinations. This region consists of 200 miles of coast north of Barcelona, from Blanes to Portbou. There’s a varied landscape too, from small fishing villages to medieval towns and mid-sized cities. If you had to choose only 2

Кажется, что ответ модели стал еще более качественным!

Что еще было бы полезно попробовать попробовать/поварьировать/потюнить:
- стратегии чанкования
- векторные индексы (ANN), и их настройки
- ретриверы, напр. гибридный ретривал
- число поднимаемых из индекса документов-кандидатов
- модели-эмбеддеры и модели-реранкеры
- промпт для LLM

Все это вы можете попробовать самостоятельно!